# Defining Parameters and utility function
## 1. Economic parameters
## 2. Numeric parameters
## 3. Constructing Grid Bounds
## 

In [14]:
using Plots
using Printf

In [15]:
Base.@kwdef struct EconomicParameters
    α::Float64 = 0.3    # capital share
    β::Float64 = 0.96   # subjective discount factor
    δ::Float64 = 1.0    # capital depreciation rate
    γ::Float64 = 1.0    # Risk Aversion rate
end 

Base.@kwdef struct NumericalParameters
    nk::Int = 200             # number of points on the capital grid, N
    kmin_rel::Float64 = 0.1   # lower grid bound, relative to k*
    kmax_rel::Float64 = 2.0   # upper grid bound, relative to k*
    spacing::Symbol = :linear # :linear or :log
    crit::Float64 = 1e-6      # convergence tolerance, ε
    maxiter::Int = 5000       # iteration cap, the safety net    
end

par = EconomicParameters()
mpar = NumericalParameters()
println("Economic parameters:  ", par)
println("Numerical parameters: ", mpar)

Economic parameters:  EconomicParameters(0.3, 0.96, 1.0, 1.0)
Numerical parameters: NumericalParameters(200, 0.1, 2.0, :linear, 1.0e-6, 5000)


In [16]:
# Utility function
function util(c, p::EconomicParameters)
    if p.γ == 1.0
        return log(c)
    else
        return (c^(1 - p.γ) - 1) / (1 - p.γ)
    end
end

util (generic function with 1 method)

In [17]:
# Computing bounds and range of grid
kstar = ((par.α) / ((1 / par.β) - 1 + par.δ))^(1 / (1 - par.α))
kmin, kmax = mpar.kmin_rel * kstar, mpar.kmax_rel * kstar
@printf("k* = %.4f, grid bounds %.4f to %.4f\n", kstar, kmin, kmax)

# Generating interior grid points
if mpar.spacing == :linear
    gri = (k = collect(range(kmin, kmax, length = mpar.nk)),)
else                                         # equidistant in ln k
    gri = (k = exp.(range(log(kmin), log(kmax), length = mpar.nk)),) # equidistant in ln k: the range runs from the log of the lower bound to the log of the upper bound
end
@printf("%d points, first spacing %.5f, last spacing %.5f\n", mpar.nk, gri.k[2] - gri.k[1], gri.k[end] - gri.k[end-1])

# Constructing actual (k,k') grid as a N x N table
meshes = (
    k      = [k  for k in gri.k, kP in gri.k],   # k_i in row i, the same in every column
    kprime = [kP for k in gri.k, kP in gri.k],   # k'_j in column j, the same in every row
)
size(meshes.k), size(meshes.kprime)

k* = 0.1689, grid bounds 0.0169 to 0.3379
200 points, first spacing 0.00161, last spacing 0.00161


((200, 200), (200, 200))

In [ ]:
# Constructing reward matrix U for baseline formulation of problem (Matrix is dependent on delta!)
Y = meshes.k .^ par.α
C = meshes.k .^ par.α - meshes.kprime .+ (1 - par.δ) .* meshes.k

U = fill(-Inf, size(C))
U[C .> 0] = util.(C[C .> 0], Ref(par))
println("share of feasible pairs: ", round(count(C .> 0) / length(C), digits = 4))
println("no NaN in U: ", !any(isnan, U))


share of feasible pairs: 0.9976
no NaN in U: true


In [19]:
# Computing closed-form benchmark for the baseline problem
F_star = par.α / (1 - par.α * par.β)
E_star = (log(1 - par.α * par.β) + par.β * F_star * log(par.α * par.β)) / (1 - par.β)
v_true = E_star .+ F_star .* log.(gri.k)
kprime_true = par.α * par.β .* gri.k .^ par.α
@printf("E = %.4f, F = %.4f\n", E_star, F_star)

E = -21.0797, F = 0.4213


In [36]:
# Value Function Iteration as a Function
function initialize_vfi(mpar)
    v      = zeros(mpar.nk)                      # line 2: the guess v_0 = 0
    g      = ones(Int, mpar.nk)                  # the policy as grid indices
    distVF = [Inf]                               # distances d_n, one per iteration
    iterVF = 0
    return timer, v, g, distVF, iterVF
end

function solve_vfi(par, mpar, U)
    v, g, distVF, iterVF = initialize_vfi(mpar)
    timer = time()
    while distVF[end] > mpar.crit && iterVF < mpar.maxiter          # line 3: keep going while the last distance, distVF[end], exceeds mpar.crit
        #global v, g, iterVF                      # the loop sits at the top level of the notebook
        M    = U .+ par.β .* v'                              # line 4: U plus β times the row vector v', the dots .+ and .* broadcast it over every row
        vals, idx = findmax(M, dims = 2)         # line 5: maximum along each row ...
        vnew = vec(vals)
        g    = getindex.(idx,2)                            # idx holds a CartesianIndex (i, j) per row: getindex.(idx, 2) picks the column j, vec turns it into a vector
        dd   = maximum(abs.(vnew - v))                              # line 6: the max norm of vnew minus v, maximum(abs.( ... ))
        v    = vnew                              # line 7: update
        iterVF += 1
        push!(distVF, dd)
    end
    time_vfi = time() - timer
    @printf("Value function iteration with parameters α = %.3f, β = %.3f, γ = %.3f, δ = %.3f, N = %.0f, crit = %.0e\n", par.α,par.β,par.γ,par.δ,mpar.nk,mpar.crit)
    @printf("iterations: %d, final distance: %.3e, time: %.3f s\n", iterVF, distVF[end], time_vfi)
end 

test = solve_vfi(par,mpar)

Value function iteration with parameters α = 0.300, β = 0.960, γ = 1.000, δ = 1.000, N = 200, crit = 1e-06
iterations: 0, final distance: 6.510e-07, time: 0.019 s


In [ ]:
# Question 2.1.a

#Grid construction 
function construct_grids(par,mpar)
    # Computing bounds and range of grid
    kstar = ((par.α) / ((1 / par.β) - 1 + par.δ))^(1 / (1 - par.α))
    kmin, kmax = mpar.kmin_rel * kstar, mpar.kmax_rel * kstar
    @printf("k* = %.4f, grid bounds %.4f to %.4f\n", kstar, kmin, kmax)

    # Generating interior grid points
    if mpar.spacing == :linear
        gri = (k = collect(range(kmin, kmax, length = mpar.nk)),)
    else                                         # equidistant in ln k
        gri = (k = exp.(range(log(kmin), log(kmax), length = mpar.nk)),) # equidistant in ln k: the range runs from the log of the lower bound to the log of the upper bound
    end
    @printf("%d points, first spacing %.5f, last spacing %.5f\n", mpar.nk, gri.k[2] - gri.k[1], gri.k[end] - gri.k[end-1])

    # Constructing actual (k,k') grid as a N x N table
    meshes = (
        k      = [k  for k in gri.k, kP in gri.k],   # k_i in row i, the same in every column
        kprime = [kP for k in gri.k, kP in gri.k],   # k'_j in column j, the same in every row
    )
    size(meshes.k), size(meshes.kprime)
    return kstar, gri, meshes
end 

# Reward matrix construction
function compute_reward(par, meshes)
    # Constructing reward matrix U for baseline formulation of problem (Matrix is dependent on delta!)
    Y = meshes.k .^ par.α
    C = meshes.k .^ par.α - meshes.kprime .+ (1 - par.δ) .* meshes.k

    U = fill(-Inf, size(C))
    U[C .> 0] = util.(C[C .> 0], Ref(par))
    println("share of feasible pairs: ", round(count(C .> 0) / length(C), digits = 4))
    println("no NaN in U: ", !any(isnan, U))
    return U
end 

function solve_vfi(par, mpar)
    @printf("Value function iteration with parameters α = %.3f, β = %.3f, γ = %.3f, δ = %.3f, N = %.0f, crit = %.0e\n", par.α,par.β,par.γ,par.δ,mpar.nk,mpar.crit)
    # import grids and reward matrix
    gri, meshes = construct_grids(par,mpar)
    U = compute_reward(par, meshes) 
    # initialize grids
    v      = zeros(mpar.nk)                      # line 2: the guess v_0 = 0
    g      = ones(Int, mpar.nk)                  # the policy as grid indices
    distVF = [Inf]                               # distances d_n, one per iteration
    iterVF = 0
    timer = time()
    while distVF[end] > mpar.crit && iterVF < mpar.maxiter          # line 3: keep going while the last distance, distVF[end], exceeds mpar.crit
        M    = U .+ par.β .* v'                              # line 4: U plus β times the row vector v', the dots .+ and .* broadcast it over every row
        vals, idx = findmax(M, dims = 2)         # line 5: maximum along each row ...
        vnew = vec(vals)
        g    = getindex.(idx,2)                            # idx holds a CartesianIndex (i, j) per row: getindex.(idx, 2) picks the column j, vec turns it into a vector
        dd   = maximum(abs.(vnew - v))                              # line 6: the max norm of vnew minus v, maximum(abs.( ... ))
        v    = vnew                              # line 7: update
        iterVF += 1
        push!(distVF, dd)
    end
    time_vfi = time() - timer
    @printf("iterations: %d, final distance: %.3e, time: %.3f s\n", iterVF, distVF[end], time_vfi)

    # validate convergence
    if distVF[end] > mpar.crit
        error("not converged: the iteration cap was hit")
    end
    kprime = gri.k[g]                            # the policy in levels, k_{g_i}
    kprime[1:5] 

    # Check for reaching grid bounds
    println("Does the policy function hit the grid bounds?")
    println("policy at the lowest grid point:  ", count(g .== 1), " states")
    println("policy at the highest grid point: ", count(g .== mpar.nk), " states")

    # Check for the Policy and Value error of the VFI
    println("Policy and Value error:")
    @printf("largest policy error: %.2e   (first grid spacing: %.2e)\n", maximum(abs.(kprime .- kprime_true)), gri.k[2] - gri.k[1])
    @printf("largest value error:  %.2e   (certificate β/(1-β) d: %.2e)\n", maximum(abs.(v .- v_true)), par.β / (1 - par.β) * distVF[end])

    # Check for the contraction rate: if should be equal to the discount rate
    println("Contraction Rate of the VFI:")
    @printf("last ratio d_n / d_(n-1): %.4f   (β = %.2f)\n", distVF[end] / distVF[end-1], par.β)
end 

test = solve_vfi(par,mpar)

Value function iteration with parameters α = 0.300, β = 0.960, γ = 1.000, δ = 1.000, N = 200, crit = 1e-06
k* = 0.1689, grid bounds 0.0169 to 0.3379
200 points, first spacing 0.00161, last spacing 0.00161
share of feasible pairs: 0.9976
no NaN in U: true
iterations: 337, final distance: 9.792e-07, time: 0.069 s
Does the policy function hit the grid bounds?
policy at the lowest grid point:  0 states
policy at the highest grid point: 0 states
Policy and Value error:
largest policy error: 9.39e-04   (first grid spacing: 1.61e-03)
largest value error:  2.41e-05   (certificate β/(1-β) d: 2.35e-05)
Contraction Rate of the VFI:
last ratio d_n / d_(n-1): 0.9600   (β = 0.96)


In [ ]:
# Question 2.1.b

# Euler equation error
#function euler_eq_error()

#end

kstar = ((par.α) / ((1 / par.β) - 1 + par.δ))^(1 / (1 - par.α))
cstar_today = (kstar)^par.α - par.δ * kstar
kprime_2 = par.α .* par.β .* meshes.k .^ par.α
cstar_next = (kprime_2)^par.α - par.δ * kprime_2
euler_error = 1 .- (par.β.*(par.α .* kprime_2 .^(par.α -1) .+ 1 .- par.δ)).^(-(1 ./ par.γ)) .* (cstar_next ./ cstar_today)


200×200 Matrix{Float64}:
 0.0706446  0.0652032  0.0596976  0.0541269  …     7.09211     6.86819
 0.0797547  0.0745216  0.0692285  0.0638746        8.69399     8.34667
 0.0879871  0.0829373  0.0778312  0.072668        11.1012     10.5205
 0.0954881  0.0906014  0.0856617  0.080668        15.144      14.0477
 0.102371   0.0976308  0.0928403  0.0879987       23.3857     20.7928
 0.108725   0.104117   0.0994618  0.0947576  …    49.5767     38.9412
 0.114622   0.110135   0.105602   0.101023      -550.186     258.671
 0.12012    0.115743   0.111323   0.106858       -42.9736    -57.4255
 0.125266   0.120992   0.116675   0.112315       -22.6024    -26.1674
 0.130102   0.125921   0.1217     0.117438       -15.4357    -17.0686
 ⋮                                           ⋱               
 0.312438   0.310843   0.309241   0.307631        -0.269205   -0.274651
 0.312769   0.311178   0.309578   0.307972        -0.266964   -0.272385
 0.313099   0.31151    0.309914   0.30831         -0.264744   -0.270